# Lab - Predictia pretului actiunilor unei companii

In [0]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import tensorflow as tf

from sklearn.preprocessing import MinMaxScaler
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, SimpleRNN

import math
from sklearn.metrics import mean_squared_error

---------------------------------------------------------------------------
ModuleNotFoundError                       Traceback (most recent call last)
File <command-2394216949479442>, line 4
      2 import matplotlib.pyplot as plt
      3 import pandas as pd
----> 4 import tensorflow as tf
      6 from sklearn.preprocessing import MinMaxScaler
      7 from tensorflow.keras.models import Sequential

File /databricks/python_shell/lib/dbruntime/autoreload/discoverability/hook.py:71, in AutoreloadDiscoverabilityHook._patched_import(self, name, *args, **kwargs)
     65 if not self._should_hint and (
     66     (module := sys.modules.get(absolute_name)) is not None and
     67     (fname := get_allowed_file_name_or_none(module)) is not None and
     68     (mtime := os.stat(fname).st_mtime) > self.last_mtime_by_modname.get(
     69         absolute_name, float("inf")) and not self._should_hint):
     70     self._should_hint = True
---> 71 module = self._original_builtins_import(name, *arg

In [0]:
!pip install tensorflow

In [0]:
%restart_python

In [0]:
file_name='IBM_2006-01-01_to_2018-01-01.csv'

In [0]:
dataset = pd.read_csv(file_name)

In [0]:
dataset = pd.read_csv(file_name, index_col='Date', parse_dates=['Date'])

In [0]:
dataset.info()

In [0]:
dataset.head()

In [0]:
dataset.tail()

In [0]:
dataset["High"][:'2016'].plot(figsize=(16,4), legend = True)
dataset["High"]['2017':].plot(figsize=(16,4), legend = True)
plt.legend(['Stock price 2006-2016', 'Stock price 2017'])
plt.show()

In [0]:
training_set=dataset[:'2016'].iloc[:, 1:2].values
training_set.shape

In [0]:
training_set[0]

In [0]:
test_set=dataset['2017':].iloc[:, 1:2].values

In [0]:
test_set[0]

In [0]:
test_set

In [0]:
test_set.shape

In [0]:
training_set[:25]

In [0]:
scaler = MinMaxScaler()


In [0]:
training_set_scalat = scaler.fit_transform(training_set)

In [0]:
training_set_scalat[:60]

In [0]:
# construim mt de training
X_train =[]
y_train =[]
for i in range(60, len(training_set_scalat)):
  X_train.append(training_set_scalat[i-60:i, 0])
  y_train.append(training_set_scalat[i,0])

In [0]:
type(X_train)

In [0]:
X_train[0]

In [0]:
y_train[0]

In [0]:
training_set_scalat[60]

In [0]:
X_train[0].shape

In [0]:
training_set_scalat[:60]

In [0]:
X_train[1]

In [0]:
y_train[0]

In [0]:
training_set_scalat[:65]

In [0]:
len(X_train)

In [0]:
len(y_train)

In [0]:
#transformam X_train si y_train din liste in np.array
X_train = np.array(X_train)
y_train = np.array(y_train)

In [0]:
X_train.shape

In [0]:
y_train.shape

In [0]:
# modificam forma X_train sa fie (2709, 60,1)
X_train=np.reshape(X_train, (X_train.shape[0], X_train.shape[1], 1))

In [0]:
X_train.shape

In [0]:
# Cream RNN
modelRNN = Sequential()

In [0]:
modelRNN.add(SimpleRNN(units = 50, input_shape=(X_train.shape[1], 1)))

In [0]:
# output layer
modelRNN.add(Dense(1))

In [0]:
modelRNN.summary()

In [0]:
modelRNN.compile(optimizer='adam', loss='mean_squared_error')

In [0]:
history = modelRNN.fit(X_train,y_train, validation_split=0.2,  epochs = 10)

In [0]:
dataset_total=pd.concat((dataset["High"][:'2016'], dataset["High"]['2017':]), axis =0)

In [0]:
dataset_total.info()

In [0]:
#3020 (total inreg) -251 (mt testare) - 60
Val_intrare = dataset_total[len(dataset_total)- len(test_set)-60:].values

In [0]:
Val_intrare.shape

In [0]:
Val_intrare = Val_intrare.reshape(-1,1)

In [0]:
 Val_intrare.shape

In [0]:
Val_intrare = scaler.transform(Val_intrare)

In [0]:
Val_intrare[:10]

In [0]:
# construim mt de testare
X_test =[]
for i in range(60, len(Val_intrare)):
  X_test.append(Val_intrare[i-60:i, 0])


In [0]:
type(X_test)

In [0]:
X_test[0]

In [0]:
X_train[-1]

In [0]:
X_test = np.array(X_test)

In [0]:
X_test.shape

In [0]:
X_test = np.reshape(X_test, (X_test.shape[0], X_test.shape[1],1))

In [0]:
X_test.shape

In [0]:
# predictia
predicted = modelRNN.predict(X_test)

In [0]:
predicted[:10]

In [0]:
test_set.shape

In [0]:
test_set[:10]

In [0]:
predicted = scaler.inverse_transform(predicted)

In [0]:
predicted[:10]

In [0]:
predicted.shape

In [0]:
test_set.shape

In [0]:
mse = math.sqrt(mean_squared_error(predicted, test_set))#root mean squared error

In [0]:
print(mse)

In [0]:
plt.plot(figuresize=(16,4))
plt.plot(predicted)
plt.plot(test_set)
plt.legend(['predicted', 'stock price'])
plt.show()